## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-05-10-49-00 +0000,wsj,How Trump’s War in Iran and Missteps Flipped W...,https://www.wsj.com/world/how-trumps-war-in-ir...
1,2026-11-05-10-45-32 +0000,nypost,"Brooklyn house fire kills 6, injures several o...",https://nypost.com/2026/11/05/us-news/nyc-hous...
2,2026-11-05-10-40-50 +0000,nypost,US Marine killed during fast-rope training in ...,https://nypost.com/2026/11/05/us-news/lance-cp...
3,2026-11-05-10-38-00 +0000,wsj,Oil Prices Rise but Stay Below $100 as Traders...,https://www.wsj.com/business/energy-oil/oil-pr...
4,2026-11-05-10-33-21 +0000,bbc,Was an Italian chef really fired over refusal ...,https://www.bbc.co.uk/news/videos/cmvg9yy8wpwe...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2652/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
0,trump,61
138,midterm,40
103,democrats,36
413,midterms,28
304,senate,22


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
270,2026-11-04-20-06-00 +0000,wsj,"Democrats won control of Congress, a majority ...",https://www.wsj.com/politics/elections/how-dem...,220
319,2026-11-04-18-06-00 +0000,wsj,"To win control of the House and Senate, Democr...",https://www.wsj.com/politics/elections/the-vot...,199
422,2026-11-04-11-34-05 +0000,chitri,Democrats win the US House and Senate as voter...,https://www.chicagotribune.com/2026/11/04/poll...,189
404,2026-11-04-12-54-34 +0000,cbc,Iranian officials welcome U.S. midterm results...,https://www.cbc.ca/news/world/iran-united-stat...,182
186,2026-11-04-22-37-00 +0000,wsj,Democrats’ midterm sweep has shaken up the 202...,https://www.wsj.com/politics/elections/democra...,181


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
270,220,2026-11-04-20-06-00 +0000,wsj,"Democrats won control of Congress, a majority ...",https://www.wsj.com/politics/elections/how-dem...
281,154,2026-11-04-19-49-18 +0000,nypost,"New Yorkers could be in charge of White House,...",https://nypost.com/2026/11/04/us-news/new-york...
186,105,2026-11-04-22-37-00 +0000,wsj,Democrats’ midterm sweep has shaken up the 202...,https://www.wsj.com/politics/elections/democra...
236,87,2026-11-04-21-21-00 +0000,wsj,Markets surged Wednesday after American voters...,https://www.wsj.com/politics/elections/the-win...
70,80,2026-11-05-06-13-59 +0000,nypost,Ted Cruz calls midterm election results a ‘Wat...,https://nypost.com/2026/11/05/us-news/ted-cruz...
190,67,2026-11-04-22-31-00 +0000,wsj,Vice President JD Vance blamed the war in Iran...,https://www.wsj.com/politics/policy/vances-bre...
260,62,2026-11-04-20-23-48 +0000,nypost,Jane Kim holds slim lead over Ben Allen with r...,https://nypost.com/2026/11/04/us-news/jane-kim...
101,43,2026-11-05-02-00-00 +0000,wsj,Americans’ votes for anti-data center candidat...,https://www.wsj.com/tech/ai/the-election-day-r...
262,40,2026-11-04-20-21-14 +0000,bbc,How voter shifts in Texas helped elect a Democ...,https://www.bbc.co.uk/news/videos/cmy4xz081kxy...
64,38,2026-11-05-07-10-50 +0000,nypost,Georgia son told 911 his missing mom was ‘abdu...,https://nypost.com/2026/11/05/us-news/georgia-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
